In [ ]:
import os

os.chdir(os.getenv("HOME_DIR"))

import gc
import time

import yaml
from albumentations import (
    CoarseDropout,
    GaussNoise,
    ReplayCompose,
    Resize,
)
from dotenv import load_dotenv
from tqdm import tqdm
from ultralytics import YOLO

from src.utils.augmentations import process_image
from src.utils.utils import convert_labels

load_dotenv()

In [ ]:
PROCESSED_DIR = os.path.join(os.getenv("HOME_DIR"), "data", "processed")
IMG_SIZE = int(os.getenv("HEIGHT")), int(os.getenv("WIDTH"))
DATASET_YAML = "bdd100k.yaml"
DATA_DIR = os.path.join(os.getenv("HOME_DIR"), "config", "datasets")
IMG_SIZE = int(os.getenv("HEIGHT")), int(os.getenv("WIDTH"))

In [ ]:
transform = ReplayCompose(
    [
        Resize(height=IMG_SIZE[0], width=IMG_SIZE[1]),
        GaussNoise(p=0.5, std_range=(0.1, 0.1)),
        CoarseDropout(p=0.5, num_holes_range=(1, 3), hole_height_range=(8, 32)),
    ]
)

# **Processing data**

In [ ]:
start = time.time()

for split in ["train", "test", "val"]:
    images_dir = os.path.join(os.getenv("RAW_IMAGES_DIR"), split)
    labels_dir = os.path.join(os.getenv("RAW_LABELS_DIR"), split)

    os.makedirs(os.path.join(PROCESSED_DIR, "images", split), exist_ok=True)
    os.makedirs(os.path.join(PROCESSED_DIR, "labels", split), exist_ok=True)
    processed_images_dir = os.path.join(PROCESSED_DIR, "images", split)
    processed_labels_dir = os.path.join(PROCESSED_DIR, "labels", split)

    selected_img_files = os.listdir(images_dir)

    batch_size_processing = 10000
    for i in range(0, len(selected_img_files), batch_size_processing):
        batch_files = selected_img_files[i : i + batch_size_processing]
        for img_file in tqdm(
            batch_files,
            desc=f"Processing {split} batch {i // batch_size_processing + 1}",
        ):
            img_path = os.path.join(images_dir, img_file)
            label_file = img_file.replace(".jpg", ".json").replace(".png", ".json")
            label_path = os.path.join(labels_dir, label_file)

            process_image(
                img_path,
                label_path,
                processed_images_dir,
                processed_labels_dir,
                transform,
                new_h=IMG_SIZE[0],
                new_w=IMG_SIZE[1],
            )

        # Clearing memory after batch
        batch_files = None
        gc.collect()

print(f"Time {time.time() - start}")

# **Training**

In [ ]:
handle_model_yaml = "yolo8_baseline.yaml"
yaml_path = os.path.join(
    os.getenv("HOME_DIR"),
    "config",
    "models",
    handle_model_yaml,
)
with open(yaml_path, "r") as file:
    args = yaml.safe_load(file)


# Set directories path for training
PROJECT_DIR = os.path.join(
    os.getenv("HOME_DIR"), "results", "models", args["project_results_name"]
)


# Modify labels from .json to .txt
for split in ["train", "val", "test"]:
    convert_labels(
        os.path.join(PROCESSED_DIR, "labels", split),
        os.path.join(PROCESSED_DIR, "labels", split),
        args["selected_classes"],
        img_size=IMG_SIZE,
    )


# Load model
resume = False
if not os.path.exists(PROJECT_DIR):
    model_path = args["model_name"]
else:
    model_path = os.path.join(PROJECT_DIR, "train", "weights", "best.pt")
    resume = True
model = YOLO(model_path, task="detect", verbose=True)


# Train
try:
    results = model.train(
        data=os.path.join(DATA_DIR, DATASET_YAML),
        project=PROJECT_DIR,
        epochs=100,
        imgsz=IMG_SIZE[0],
        batch=8,
        exist_ok=True,
        resume=resume,
        device=-1,
        patience=10,
        optimizer="AdamW",
        plots=True,
        amp=False,
    )
except Exception as e:
    print(f"Last training was finished: {e}.")

# **Results**

## **Metrics**

![results.png](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/train/results.png)
![BoxF1_curve.png](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/train/BoxF1_curve.png)

## **Predicted Images**

![val_batch0_pred.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/train/val_batch0_pred.jpg)
![val_batch1_pred.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/models/yolo8_baseline/train/val_batch1_pred.jpg)

![d50594a3-1c5bd6fa.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/visualizations/yolo8_baseline/comparison/d50594a3-1c5bd6fa.jpg)
![ed4a69c0-c18344c7.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/visualizations/yolo8_baseline/comparison/ed4a69c0-c18344c7.jpg)
![f20bfbfd-f86e430e.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/visualizations/yolo8_baseline/comparison/f20bfbfd-f86e430e.jpg)

![d50594a3-1c5bd6fa.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/visualizations/yolo8_baseline/predict/d50594a3-1c5bd6fa.jpg)
![ed4a69c0-c18344c7.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/visualizations/yolo8_baseline/predict/ed4a69c0-c18344c7.jpg)
![f20bfbfd-f86e430e.jpg](/home/borealis/Documents/study/Netology/dll/final_project_dll/Real-Time-Detection/results/visualizations/yolo8_baseline/predict/f20bfbfd-f86e430e.jpg)